In [1]:
!pip install wandb

In [ ]:
import wandb

with wandb.init(
    entity="YacssoK",
    project="practice_sun",
    name="1101_dataB_yolo26s",
    tags=["rfs"],
    config = "configs/yolo26s.yaml"
) as run:
    run.log({"connection_test": 1})

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


In [ ]:
import os, sys, csv, wandb
from pathlib import Path
ROOT = Path("../develop").expanduser()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.utils import load_config, resolve_path
from src.interfaces import MAX_PILLS_PER_IMAGE
from ultralytics import YOLO

In [ ]:


cfg = load_config("configs/yolo26s.yaml") 
run_dir = ROOT / "outputs/1101_dataB_yolo26s_v0_/yolo_run"
data_yaml = resolve_path(cfg["data"]["yolo_dir"]) / cfg["data"]["version"] / "data.yaml"
ev = cfg.get("eval", {}) or {}

def map75_95(split):
    m = YOLO(str(run_dir / "weights/best.pt")).val(
        data=str(data_yaml), split=split, imgsz=cfg["data"]["img_size"],
        batch=cfg["train"]["batch_size"], device="mps", plots=True, verbose=False,
        conf=ev.get("conf_thr", 0.001), iou=ev.get("nms_iou", 0.5),
        max_det=ev.get("max_det", MAX_PILLS_PER_IMAGE))
    return float(m.box.all_ap[:, 5:].mean())

with wandb.init(
    entity="YacssoK",
    project="practice_sun",
    name="110-1_dataB_yolo26s_v0_rfs01",
    group="train",
    tags=["yolo26s", "rfs"],
    config=cfg,
) as run:
    for r in csv.DictReader(open(run_dir / "results.csv", encoding="utf-8")):
        r = {k.strip(): v for k, v in r.items()}
        run.log({k: float(v) for k, v in r.items() if "loss" in k}, step=int(float(r["epoch"])))
    print("loss 업로드 끝, mAP75_95 채점 중...")
    tr, va = map75_95("train"), map75_95("val")
    run.summary.update({"train_mAP75_95": tr, "val_mAP75_95": va, "gap_mAP75_95": tr - va})
    print(f"train {tr:.4f} / val {va:.4f}")

loss 업로드 끝, mAP75_95 채점 중...
Ultralytics 8.4.183 🚀 Python-3.11.17 torch-2.14.1 MPS (Apple M3)
Model summary (fused): 72 layers, 11,147,256 parameters, 0 gradients, 28.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1080.8±302.2 MB/s, size: 1669.5 KB)
val: Scanning /Users/pool/Desktop/develop/data/yolo/v0/labels/train.cache... 185 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 185/185 194.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 1.5it/s 8.0s0.7s
                   all        185        610      0.991      0.995      0.993      0.991
Speed: 0.2ms preprocess, 2.8ms inference, 0.0ms loss, 17.4ms postprocess per image
Results saved to /Users/pool/Desktop/Beginner_Project/runs/detect/val-3
Ultralytics 8.4.183 🚀 Python-3.11.17 torch-2.14.1 MPS (Apple M3)
Model summary (fused): 72 layers, 11,147,256 parameters, 0 gradients, 28.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1379.7±21

train/box_loss,█▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train/cls_loss,█▅▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/dfl_loss,█▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
val/box_loss,█▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁
val/cls_loss,█▆▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/dfl_loss,█▅▅▄▄▄▃▂▂▂▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁
gap_mAP75_95,0.03681
train/box_loss,0.18864
train/cls_loss,0.25403
train/dfl_loss,0.78088
train_mAP75_95,0.98877


In [ ]:
cfg = load_config("configs/yolo_img1024.yaml")
run_dir = ROOT / "outputs/1024_sun_yolov8s_v0_img1024/yolo_run"
data_yaml = resolve_path(cfg["data"]["yolo_dir"]) / cfg["data"]["version"] / "data.yaml"
ev = cfg.get("eval", {}) or {}

def map75_95(split):
    m = YOLO(str(run_dir / "weights/best.pt")).val(
        data=str(data_yaml), split=split, imgsz=cfg["data"]["img_size"],
        batch=cfg["train"]["batch_size"], device="mps", plots=False, verbose=False,
        conf=ev.get("conf_thr", 0.001), iou=ev.get("nms_iou", 0.5),
        max_det=ev.get("max_det", MAX_PILLS_PER_IMAGE))
    return float(m.box.all_ap[:, 5:].mean())

with wandb.init(
    entity="YacssoK",
    project="practice_sun",
    name="1024_sun_yolov8s_v0_img1024",
    group="train",
    tags=["yolov8s", "1024"],
    config=cfg,
) as run:
    for r in csv.DictReader(open(run_dir / "results.csv", encoding="utf-8")):
        r = {k.strip(): v for k, v in r.items()}
        run.log({k: float(v) for k, v in r.items() if "loss" in k}, step=int(float(r["epoch"])))
    print("loss 업로드 끝, mAP75_95 채점 중...")
    tr, va = map75_95("train"), map75_95("val")
    run.summary.update({"train_mAP75_95": tr, "val_mAP75_95": va, "gap_mAP75_95": tr - va})
    print(f"train {tr:.4f} / val {va:.4f}")

NameError: name 'load_config' is not defined

In [ ]:
 
# wandb.init(
#     project="rfc",
#     entity="YacssoK",
#     name="rfc01"
# )

In [ ]:
# wandb.log({"test": 1})

In [ ]:
#uv sync --locked --extra tracking --extra demo

In [ ]:
#uv run --locked --extra tracking --extra demo python -c "import wandb; print(wandb.__version__)"#

In [ ]:
#uv add wandb

In [ ]:
#uv run wandb login

In [ ]:
# import os

In [ ]:
# project_id = "practice_sun"

# with wandb.init(project=project_id, config = config) as run:
     